# Robust data center exposure calculation with capacity scenarios

This notebook preserves every exposure
measure and robustness variable created by robust3 and adds capacity-adjusted
data center exposure (DCE) measures for six user-defined capacity scenarios
(`capacity1`-`capacity6`).

Capacity-adjusted DCE is calculated only within a 25 km buffer using two
functional forms:

1. **Capacity-distance exposure**: data center capacity multiplied by the
   baseline inverse-distance weight. For data center \(j\) and coal unit \(i\),
   the contribution is
   `capacity_j / max(distance_ij_10km(unit), 0.1)`.
2. **Capacity-sum exposure**: the sum of data center capacity within 25 km,
   without inverse-distance weighting.

For each capacity scenario and functional form, the notebook creates:

- exposure to all data centers;
- exposure to larger data centers under the baseline classification
  (hyperscale, cloud, wholesale and crypto-mining);
- exposure to all other data centers;
- four commissioning-period measures (`before06`, `06_15`, `16_19`,
  `20_24`); and
- one measure aggregated across all commissioning periods (`all`).



In [ ]:
import os
import re
import time
import warnings
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import Point
from tqdm import tqdm

warnings.filterwarnings('ignore')

current = os.getcwd()
while os.path.basename(current) != "Data_center_and_fossil_energy_Replication":
    parent = os.path.dirname(current)
    if parent == current:
        raise RuntimeError("Could not find Data_center_and_fossil_energy_Replication root")
    current = parent

BASE_PATH = current
RAW = os.path.join(BASE_PATH, 'Data', 'raw')
TEMP = os.path.join(BASE_PATH, 'Data', 'temp')
USE = os.path.join(BASE_PATH, 'Data', 'use')
FIGURES = os.path.join(BASE_PATH, 'Results', 'Figures')
TABLES = os.path.join(BASE_PATH, 'Results', 'Tables')

for path in [RAW, TEMP, USE, FIGURES, TABLES]:
    os.makedirs(path, exist_ok=True)

print(f"BASE_PATH: {BASE_PATH}")


In [2]:
COAL_INPUT = os.path.join(TEMP, "gem_coal_plants_multi_record_sa_sinceoperating.dta")
AI_INPUT = os.path.join(RAW, "SPGlobal_Export.xlsx")
GADM_INPUT = os.path.join(RAW, "gadm_410.gpkg")
ELECZONE_INPUT = os.path.join(RAW, "world.geojson")

OUTPUT_DTA = os.path.join(TEMP, "gem_coal_plants_multi_record_sa_dce_robust3.dta")
VAR_DICT_OUTPUT = os.path.join(TEMP, "dce_robust3_variable_dictionary.csv")
SUMMARY_OUTPUT = os.path.join(TEMP, "dce_robust3_summary_statistics.csv")

BASE_WINDOWS = {
    'before06': (None, 2005),
    '06_15': (2006, 2015),
    '16_19': (2016, 2019),
    '20_24': (2020, 2024),
}

BUFFER_DISTANCES = [15, 25, 50, 100, 200]
TEMPORAL_CUTOFFS = [2018, 2019, 2021, 2022]
TEMPORAL_CUTOFF_BUFFERS = [25]

# Additional timing test: one mutually exclusive multi-period
# specification isolates each commissioning year from 2018 to 2022.
ANNUAL_WINDOWS = {
    'before06': (None, 2005),
    '06_15': (2006, 2015),
    '16_17': (2016, 2017),
    '2018': (2018, 2018),
    '2019': (2019, 2019),
    '2020': (2020, 2020),
    '2021': (2021, 2021),
    '2022': (2022, 2022),
    '23_24': (2023, 2024),
}
ANNUAL_WINDOW_BUFFERS = [25]
MIN_DISTANCE_CAPS_KM = [1, 2, 3]
MIN_CAP_BUFFERS = [25]
WINSOR_QUANTILES = [0.95, 0.99]
CRYPTO_TYPE = 'Crypto Mining Data Center'
WHOLESALE_TYPE = 'Wholesale Data Center'

# Larger/other data-center split follows large_scale_dc_exposure_calculate.ipynb.
# Larger includes hyperscale, cloud, wholesale, and crypto-mining data centers.
LARGER_DC_TYPES = [
    'Hyperscale Data Center',
    'Cloud Data Center',
    'Wholesale Data Center',
    'Crypto Mining Data Center',
]

# Alternative larger/other classifications for robustness.
# 1. No-crypto larger: crypto mining is moved to other.
LARGER_DC_TYPES_NO_CRYPTO = [
    'Hyperscale Data Center',
    'Cloud Data Center',
    'Wholesale Data Center',
]

# 2. Strict larger: only hyperscale and cloud are classified as larger;
#    both crypto mining and wholesale data centers are moved to other.
LARGER_DC_TYPES_STRICT = [
    'Hyperscale Data Center',
    'Cloud Data Center',
]

CAPACITY_COLUMNS = {
    1: 'capacity1',
    2: 'capacity2',
    3: 'capacity3',
    4: 'capacity4',
    5: 'capacity5',
    6: 'capacity6',
}

# Capacity-adjusted measures are calculated only within 25 km.
CAPACITY_BUFFER_KM = 25
CAPACITY_MIN_DISTANCE_KM = 0.1

# In addition to the four mutually exclusive baseline periods, "all" pools
# all data centers commissioned through 2024 while retaining the same
# cumulative-by-panel-year exposure logic.
CAPACITY_WINDOWS = {
    **BASE_WINDOWS,
    'all': (None, 2024),
}

# Short prefixes keep every generated variable within Stata's 32-character
# variable-name limit:
#   dw  = capacity x inverse-distance weight
#   cs  = unweighted capacity sum within 25 km
#   l   = larger data centers
#   o   = other data centers

LARGER_OTHER_CLASSIFICATIONS = [
    {
        'label': 'baseline',
        'larger_prefix': 'ai_larger',
        'other_prefix': 'ai_otherdc',
        'cut_larger_prefix': 'ai_larger_cut',
        'cut_other_prefix': 'ai_otherdc_cut',
        'annual_larger_prefix': 'ai_lgann',
        'annual_other_prefix': 'ai_otann',
        'larger_types': LARGER_DC_TYPES,
        'note': 'larger includes hyperscale, cloud, wholesale, and crypto mining data centers'
    },
    {
        'label': 'no_crypto_larger',
        'larger_prefix': 'ai_largenc',
        'other_prefix': 'ai_othernc',
        'cut_larger_prefix': 'ai_lgnc_cut',
        'cut_other_prefix': 'ai_otnc_cut',
        'annual_larger_prefix': 'ai_lnann',
        'annual_other_prefix': 'ai_onann',
        'larger_types': LARGER_DC_TYPES_NO_CRYPTO,
        'note': 'crypto mining data centers are moved from larger to other; larger includes hyperscale, cloud, and wholesale data centers'
    },
    {
        'label': 'strict_larger',
        'larger_prefix': 'ai_largest',
        'other_prefix': 'ai_otherst',
        'cut_larger_prefix': 'ai_lgst_cut',
        'cut_other_prefix': 'ai_otst_cut',
        'annual_larger_prefix': 'ai_lsann',
        'annual_other_prefix': 'ai_osann',
        'larger_types': LARGER_DC_TYPES_STRICT,
        'note': 'strict larger includes only hyperscale and cloud data centers; crypto mining and wholesale data centers are moved to other'
    },
]


In [3]:
def calculate_distance_vectorized(lat1, lon1, lat2_array, lon2_array):
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2_array)
    lon2_rad = np.radians(lon2_array)
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c


def normalize_text(x):
    if pd.isna(x):
        return ''
    x = str(x).strip().lower()
    x = re.sub(r'[^a-z0-9]+', ' ', x)
    x = re.sub(r'\s+', ' ', x).strip()
    return x


def first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


def make_country_key(df, preferred=None):
    candidates = []
    if preferred:
        candidates.append(preferred)
    candidates += ['Country/Area', 'Country', 'COUNTRY', 'country', 'Alpha-3 code', 'GID_0', 'NAME_0']
    col = first_existing_column(df, candidates)
    if col is None:
        return pd.Series([''] * len(df), index=df.index), None
    return df[col].map(normalize_text), col


def exposure_from_years(ai_year_subset, weight_subset, target_years):
    if len(ai_year_subset) == 0:
        return np.zeros(len(target_years), dtype=float)
    order = np.argsort(ai_year_subset)
    years_sorted = ai_year_subset[order]
    weights_cum = np.cumsum(weight_subset[order])
    pos = np.searchsorted(years_sorted, target_years, side='right') - 1
    out = np.zeros(len(target_years), dtype=float)
    valid = pos >= 0
    out[valid] = weights_cum[pos[valid]]
    return out


def time_window_mask(ai_year, start_year, end_year):
    if start_year is None:
        return ai_year <= end_year
    return (ai_year >= start_year) & (ai_year <= end_year)


def safe_stata_varname(name):
    name = re.sub(r'[^A-Za-z0-9_]', '_', name)
    if len(name) > 32:
        raise ValueError(f"Variable name exceeds Stata 32-char limit: {name} ({len(name)})")
    return name


In [4]:
def spatial_join_gadm_points(df, lat_col, lon_col, gadm_gdf, id_cols=None, label='points'):
    id_cols = id_cols or []
    keep_cols = list(dict.fromkeys(id_cols + [lat_col, lon_col]))
    point_df = df[keep_cols].dropna(subset=[lat_col, lon_col]).copy()
    geometry = [Point(xy) for xy in zip(point_df[lon_col], point_df[lat_col])]
    point_gdf = gpd.GeoDataFrame(point_df, geometry=geometry, crs='EPSG:4326')

    if gadm_gdf.crs != point_gdf.crs:
        gadm_gdf = gadm_gdf.to_crs(point_gdf.crs)

    print(f"  - Spatial joining {label}: {len(point_gdf):,} points")
    joined = gpd.sjoin(point_gdf, gadm_gdf, how='left', predicate='within')
    matched = joined['index_right'].notna().sum() if 'index_right' in joined.columns else 0
    print(f"    matched: {matched:,}/{len(joined):,} ({matched / len(joined) * 100:.1f}%)")

    gadm_cols = [
        'GID_0', 'NAME_0', 'GID_1', 'NAME_1', 'GID_2', 'NAME_2',
        'COUNTRY', 'CONTINENT', 'REGION', 'SUBCONT'
    ]
    available = [c for c in gadm_cols if c in joined.columns]
    return pd.DataFrame(joined[id_cols + available]) if id_cols else pd.DataFrame(joined[available])


def ensure_gadm_attributes(df_coal, df_ai):
    print("\nLoading GADM boundaries...")
    gadm_gdf = gpd.read_file(GADM_INPUT)
    print(f"  GADM records: {len(gadm_gdf):,}")

    needed = ['GID_1', 'GID_2', 'NAME_1', 'NAME_2']
    if not all(c in df_coal.columns for c in needed):
        unique_units = df_coal[['GEM_unit_phase_ID', 'Latitude', 'Longitude']].drop_duplicates('GEM_unit_phase_ID').copy()
        coal_gadm = spatial_join_gadm_points(
            unique_units, 'Latitude', 'Longitude', gadm_gdf,
            id_cols=['GEM_unit_phase_ID'], label='unique coal units'
        )
        rename_map = {c: f'{c}_gadm' for c in ['COUNTRY', 'REGION'] if c in coal_gadm.columns}
        coal_gadm = coal_gadm.rename(columns=rename_map)
        df_coal = df_coal.merge(coal_gadm, on='GEM_unit_phase_ID', how='left')
    else:
        print("\nCoal data already contain GID_1/GID_2/NAME_1/NAME_2; skipping coal GADM join.")

    ai_gadm = spatial_join_gadm_points(
        df_ai.reset_index().rename(columns={'index': '_ai_row_id'}),
        'LATITUDE', 'LONGITUDE', gadm_gdf,
        id_cols=['_ai_row_id'], label='data centers'
    )
    ai_gadm = ai_gadm.rename(columns={
        'GID_0': 'ai_GID_0', 'NAME_0': 'ai_NAME_0',
        'GID_1': 'ai_GID_1', 'NAME_1': 'ai_NAME_1',
        'GID_2': 'ai_GID_2', 'NAME_2': 'ai_NAME_2',
        'COUNTRY': 'ai_COUNTRY_gadm', 'REGION': 'ai_REGION_gadm',
        'CONTINENT': 'ai_CONTINENT', 'SUBCONT': 'ai_SUBCONT'
    })
    df_ai = df_ai.reset_index().rename(columns={'index': '_ai_row_id'}).merge(ai_gadm, on='_ai_row_id', how='left')
    return df_coal, df_ai

def spatial_join_electricity_zones(
    df,
    lat_col,
    lon_col,
    zones_gdf,
    id_col,
    label='points',
):
    """
    Assign each point to one Electricity Maps zone.

    If zone polygons overlap, retain the smallest-area polygon, which is
    generally the more spatially specific zone. Points not matched with
    predicate='within' are retried using predicate='intersects' so that
    points located exactly on polygon boundaries are not lost.
    """
    point_df = (
        df[[id_col, lat_col, lon_col]]
        .dropna(subset=[lat_col, lon_col])
        .drop_duplicates(subset=[id_col])
        .copy()
    )
    point_gdf = gpd.GeoDataFrame(
        point_df,
        geometry=gpd.points_from_xy(
            point_df[lon_col],
            point_df[lat_col],
        ),
        crs='EPSG:4326',
    )

    if zones_gdf.crs != point_gdf.crs:
        zones_gdf = zones_gdf.to_crs(point_gdf.crs)

    zone_cols = [
        'zoneName',
        'countryKey',
        'countryName',
        '_zone_area_km2',
        'geometry',
    ]
    zones_use = zones_gdf[zone_cols].copy()

    print(f"  - Spatial joining {label}: {len(point_gdf):,} points")
    joined = gpd.sjoin(
        point_gdf,
        zones_use,
        how='left',
        predicate='within',
    )

    matched_ids = set(
        joined.loc[joined['zoneName'].notna(), id_col]
    )
    unmatched = point_gdf.loc[
        ~point_gdf[id_col].isin(matched_ids)
    ].copy()

    if len(unmatched) > 0:
        boundary_join = gpd.sjoin(
            unmatched,
            zones_use,
            how='left',
            predicate='intersects',
        )
        joined = pd.concat(
            [
                joined.loc[~joined[id_col].isin(unmatched[id_col])],
                boundary_join,
            ],
            ignore_index=True,
        )

    # Overlapping polygons can produce more than one match. Prefer the
    # smallest zone polygon and then zoneName for deterministic selection.
    joined['_matched'] = joined['zoneName'].notna()
    joined['_sort_area'] = joined['_zone_area_km2'].fillna(np.inf)
    joined = (
        joined.sort_values(
            [id_col, '_matched', '_sort_area', 'zoneName'],
            ascending=[True, False, True, True],
            na_position='last',
        )
        .drop_duplicates(subset=[id_col], keep='first')
    )

    matched = int(joined['zoneName'].notna().sum())
    print(
        f"    matched: {matched:,}/{len(point_gdf):,} "
        f"({matched / len(point_gdf) * 100:.1f}%)"
    )

    return pd.DataFrame(
        joined[[id_col, 'zoneName', 'countryKey', 'countryName']]
    )


def ensure_electricity_zone_attributes(df_coal, df_ai):
    print("\nLoading Electricity Maps zone boundaries...")
    zones_gdf = gpd.read_file(ELECZONE_INPUT)

    # Repair invalid polygons before spatial matching.
    invalid_n = int((~zones_gdf.geometry.is_valid).sum())
    if invalid_n > 0:
        print(f"  Repairing {invalid_n} invalid zone geometries")
        zones_gdf['geometry'] = zones_gdf.geometry.make_valid()

    zones_gdf = zones_gdf.loc[
        zones_gdf.geometry.notna() & ~zones_gdf.geometry.is_empty
    ].copy()

    # Equal-area projection is used only to resolve overlapping polygons.
    zones_equal_area = zones_gdf.to_crs('EPSG:6933')
    zones_gdf['_zone_area_km2'] = (
        zones_equal_area.geometry.area.to_numpy() / 1_000_000
    )

    print(
        f"  Electricity zones: {len(zones_gdf):,}; "
        f"countries/areas: {zones_gdf['countryKey'].nunique():,}"
    )

    unique_units = (
        df_coal[
            ['GEM_unit_phase_ID', 'Latitude', 'Longitude']
        ]
        .drop_duplicates('GEM_unit_phase_ID')
        .copy()
    )
    coal_zones = spatial_join_electricity_zones(
        unique_units,
        'Latitude',
        'Longitude',
        zones_gdf,
        id_col='GEM_unit_phase_ID',
        label='unique coal units',
    ).rename(
        columns={
            'zoneName': 'electricity_zone',
            'countryKey': 'electricity_country',
            'countryName': 'electricity_country_name',
        }
    )
    df_coal = df_coal.merge(
        coal_zones,
        on='GEM_unit_phase_ID',
        how='left',
        validate='many_to_one',
    )

    if '_ai_row_id' not in df_ai.columns:
        df_ai = df_ai.reset_index(drop=True).copy()
        df_ai['_ai_row_id'] = np.arange(len(df_ai))

    ai_zones = spatial_join_electricity_zones(
        df_ai,
        'LATITUDE',
        'LONGITUDE',
        zones_gdf,
        id_col='_ai_row_id',
        label='data centers',
    ).rename(
        columns={
            'zoneName': 'ai_electricity_zone',
            'countryKey': 'ai_electricity_country',
            'countryName': 'ai_electricity_country_name',
        }
    )
    df_ai = df_ai.merge(
        ai_zones,
        on='_ai_row_id',
        how='left',
        validate='one_to_one',
    )

    return df_coal, df_ai



In [5]:
def calculate_buffer_exposures(df_coal, df_ai, buffer_distances, time_windows,
                               prefix='ai_proximity', min_distance_km=0.1,
                               ai_filter=None, variable_notes=None):
    variable_notes = variable_notes if variable_notes is not None else []
    ai_sub = df_ai.copy()
    if ai_filter is not None:
        ai_sub = ai_sub.loc[ai_filter(ai_sub)].copy()

    ai_lat = ai_sub['LATITUDE'].to_numpy(float)
    ai_lon = ai_sub['LONGITUDE'].to_numpy(float)
    ai_year = ai_sub['YR_BUILT'].to_numpy(int)

    new_cols = []
    for distance_km in buffer_distances:
        for window_name in time_windows:
            col = safe_stata_varname(f'{prefix}_{distance_km}km_{window_name}')
            df_coal[col] = 0.0
            new_cols.append(col)
            variable_notes.append({
                'variable': col,
                'type': 'buffer',
                'prefix': prefix,
                'buffer_km': distance_km,
                'time_window': window_name,
                'min_distance_km': min_distance_km,
                'notes': 'inverse distance weighted DCE within circular buffer'
            })

    print(f"\nCalculating buffer exposures: prefix={prefix}, AI centers={len(ai_sub):,}, variables={len(new_cols)}")
    grouped = df_coal.groupby('GEM_unit_phase_ID', sort=False)
    start_time = time.time()

    for n, (gem_id, group_df) in enumerate(tqdm(grouped, desc=f'buffer {prefix}', ncols=90), 1):
        coal_lat = group_df['Latitude'].iloc[0]
        coal_lon = group_df['Longitude'].iloc[0]
        target_years = group_df['year'].astype(int).to_numpy()
        distances_km = calculate_distance_vectorized(coal_lat, coal_lon, ai_lat, ai_lon)
        weights = 1.0 / (np.maximum(distances_km, min_distance_km) / 10.0)

        for distance_km in buffer_distances:
            within_buffer = distances_km <= distance_km
            if not within_buffer.any():
                continue
            for window_name, (start_year, end_year) in time_windows.items():
                col = f'{prefix}_{distance_km}km_{window_name}'
                valid = within_buffer & time_window_mask(ai_year, start_year, end_year)
                vals = exposure_from_years(ai_year[valid], weights[valid], target_years)
                df_coal.loc[group_df.index, col] = vals

        if n % 1000 == 0:
            elapsed = (time.time() - start_time) / 60
            print(f"  processed {n:,} units; elapsed {elapsed:.1f} min")

    return new_cols


In [6]:
def load_capacity_scenarios(base_df):
    """Validate and prepare capacity1-capacity6 from the Sheet1 sample."""
    capacity_cols = list(CAPACITY_COLUMNS.values())
    missing_cols = [
        col for col in capacity_cols
        if col not in base_df.columns
    ]

    if missing_cols:
        raise KeyError(
            "Missing capacity columns in Sheet1: "
            + ", ".join(missing_cols)
        )

    out = base_df.copy()

    for col in capacity_cols:
        out[col] = pd.to_numeric(out[col], errors='coerce')

    print("  Capacity scenarios loaded directly from worksheet: Sheet1")
    for scenario, col in CAPACITY_COLUMNS.items():
        valid_n = int(out[col].notna().sum())
        positive_n = int((out[col] > 0).sum())
        print(
            f"  {col}: nonmissing={valid_n:,}/{len(out):,}; "
            f"positive={positive_n:,}"
        )

    return out


def calculate_capacity_buffer_exposures(
    df_coal,
    df_ai,
    capacity_col,
    time_windows,
    prefix,
    method,
    buffer_km=25,
    min_distance_km=0.1,
    ai_filter=None,
    variable_notes=None,
):
    """
    Calculate capacity-adjusted DCE within a circular buffer.

    method='distance_capacity':
        capacity * 10 / max(distance_km, min_distance_km)

    method='capacity_sum':
        capacity, summed across eligible data centers within the buffer
    """
    if method not in {'distance_capacity', 'capacity_sum'}:
        raise ValueError(f"Unknown capacity exposure method: {method}")

    variable_notes = variable_notes if variable_notes is not None else []
    ai_sub = df_ai.copy()

    if ai_filter is not None:
        ai_sub = ai_sub.loc[ai_filter(ai_sub)].copy()

    ai_sub = ai_sub.dropna(
        subset=['LATITUDE', 'LONGITUDE', 'YR_BUILT', capacity_col]
    ).copy()
    ai_sub = ai_sub.loc[ai_sub[capacity_col] >= 0].copy()

    ai_lat = ai_sub['LATITUDE'].to_numpy(float)
    ai_lon = ai_sub['LONGITUDE'].to_numpy(float)
    ai_year = ai_sub['YR_BUILT'].to_numpy(int)
    ai_capacity = ai_sub[capacity_col].to_numpy(float)

    new_cols = []
    for window_name in time_windows:
        col = safe_stata_varname(
            f'{prefix}_{buffer_km}km_{window_name}'
        )
        df_coal[col] = 0.0
        new_cols.append(col)
        variable_notes.append({
            'variable': col,
            'type': 'capacity_adjusted_buffer',
            'prefix': prefix,
            'capacity_scenario': capacity_col,
            'capacity_method': method,
            'buffer_km': buffer_km,
            'time_window': window_name,
            'min_distance_km': (
                min_distance_km
                if method == 'distance_capacity'
                else np.nan
            ),
            'notes': (
                'capacity multiplied by baseline inverse-distance weight '
                'within 25 km'
                if method == 'distance_capacity'
                else 'sum of data center capacity within 25 km without '
                     'distance weighting'
            ),
        })

    print(
        f"\nCalculating capacity exposures: prefix={prefix}, "
        f"scenario={capacity_col}, method={method}, "
        f"AI centers={len(ai_sub):,}, variables={len(new_cols)}"
    )

    grouped = df_coal.groupby('GEM_unit_phase_ID', sort=False)

    for gem_id, group_df in tqdm(
        grouped,
        desc=f'capacity {prefix}',
        ncols=90
    ):
        coal_lat = group_df['Latitude'].iloc[0]
        coal_lon = group_df['Longitude'].iloc[0]
        target_years = group_df['year'].astype(int).to_numpy()

        distances_km = calculate_distance_vectorized(
            coal_lat,
            coal_lon,
            ai_lat,
            ai_lon
        )
        within_buffer = distances_km <= buffer_km

        if not within_buffer.any():
            continue

        if method == 'distance_capacity':
            contribution = ai_capacity * (
                1.0 / (
                    np.maximum(distances_km, min_distance_km) / 10.0
                )
            )
        else:
            contribution = ai_capacity.copy()

        for window_name, (start_year, end_year) in time_windows.items():
            valid = (
                within_buffer
                & time_window_mask(ai_year, start_year, end_year)
            )
            vals = exposure_from_years(
                ai_year[valid],
                contribution[valid],
                target_years
            )
            col = f'{prefix}_{buffer_km}km_{window_name}'
            df_coal.loc[group_df.index, col] = vals

    return new_cols


def check_capacity_larger_other_consistency(df, capacity_specs, periods):
    checks = []

    for spec in capacity_specs:
        for period in periods:
            total_col = f"{spec['total_prefix']}_25km_{period}"
            larger_col = f"{spec['larger_prefix']}_25km_{period}"
            other_col = f"{spec['other_prefix']}_25km_{period}"

            if all(
                col in df.columns
                for col in [total_col, larger_col, other_col]
            ):
                difference = (
                    df[total_col]
                    - df[larger_col]
                    - df[other_col]
                )
                checks.append({
                    'capacity_scenario': spec['capacity_col'],
                    'capacity_method': spec['method'],
                    'period': period,
                    'total_variable': total_col,
                    'larger_variable': larger_col,
                    'other_variable': other_col,
                    'max_abs_difference': float(
                        difference.abs().max()
                    ),
                })

    checks_df = pd.DataFrame(checks)
    if len(checks_df) > 0:
        print("\nCapacity larger + other consistency checks:")
        print(checks_df.to_string(index=False))

    return checks_df


In [7]:
def add_region_keys(df_coal, df_ai):
    # Same-region robustness uses only formal GADM identifiers.
    # Logic: for each coal unit, count DCE from data centers located in the same GID_1 or GID_2.
    # We intentionally do not use Subnational_unit_province_state because it is not a harmonized field on both sides.
    df_coal['key_gid1'] = df_coal['GID_1'].fillna('').astype(str)
    df_coal['key_gid2'] = df_coal['GID_2'].fillna('').astype(str)
    df_ai['key_gid1'] = df_ai['ai_GID_1'].fillna('').astype(str)
    df_ai['key_gid2'] = df_ai['ai_GID_2'].fillna('').astype(str)
    df_coal['key_ezone'] = (
        df_coal['electricity_zone'].fillna('').astype(str)
    )
    df_ai['key_ezone'] = (
        df_ai['ai_electricity_zone'].fillna('').astype(str)
    )

    print(
        "Same-region keys prepared: GID_1, GID_2, "
        "and Electricity Maps zone."
    )
    print("Subnational_unit_province_state is not used because it is not harmonized for data centers.")
    return df_coal, df_ai

def calculate_region_exposures(df_coal, df_ai, region_specs, time_windows,
                               min_distance_km=0.1, ai_filter=None, variable_notes=None):
    variable_notes = variable_notes if variable_notes is not None else []
    ai_sub = df_ai.copy()
    if ai_filter is not None:
        ai_sub = ai_sub.loc[ai_filter(ai_sub)].copy()
    ai_sub = ai_sub.reset_index(drop=True)

    ai_lat_all = ai_sub['LATITUDE'].to_numpy(float)
    ai_lon_all = ai_sub['LONGITUDE'].to_numpy(float)
    ai_year_all = ai_sub['YR_BUILT'].to_numpy(int)

    new_cols = []
    for prefix, coal_key_col, ai_key_col in region_specs:
        for window_name in time_windows:
            col = safe_stata_varname(f'{prefix}_{window_name}')
            df_coal[col] = 0.0
            new_cols.append(col)
            variable_notes.append({
                'variable': col,
                'type': 'same_region',
                'region_definition': prefix,
                'coal_key_col': coal_key_col,
                'ai_key_col': ai_key_col,
                'time_window': window_name,
                'min_distance_km': min_distance_km,
                'notes': 'inverse distance weighted DCE among data centers sharing the same region key; no circular buffer'
            })

    print(f"\nCalculating same-region exposures: AI centers={len(ai_sub):,}, variables={len(new_cols)}")
    grouped = df_coal.groupby('GEM_unit_phase_ID', sort=False)
    ai_index_by_key = {ai_key_col: ai_sub.groupby(ai_key_col).indices for _, _, ai_key_col in region_specs}

    for gem_id, group_df in tqdm(grouped, desc='same-region exposures', ncols=90):
        coal_lat = group_df['Latitude'].iloc[0]
        coal_lon = group_df['Longitude'].iloc[0]
        target_years = group_df['year'].astype(int).to_numpy()

        for prefix, coal_key_col, ai_key_col in region_specs:
            key = group_df[coal_key_col].iloc[0] if coal_key_col in group_df.columns else ''
            if key == '' or pd.isna(key):
                continue
            ai_indices = ai_index_by_key[ai_key_col].get(key, [])
            if len(ai_indices) == 0:
                continue

            ai_indices = np.array(list(ai_indices), dtype=int)
            distances_km = calculate_distance_vectorized(coal_lat, coal_lon, ai_lat_all[ai_indices], ai_lon_all[ai_indices])
            weights = 1.0 / (np.maximum(distances_km, min_distance_km) / 10.0)
            ai_year = ai_year_all[ai_indices]

            for window_name, (start_year, end_year) in time_windows.items():
                valid = time_window_mask(ai_year, start_year, end_year)
                vals = exposure_from_years(ai_year[valid], weights[valid], target_years)
                df_coal.loc[group_df.index, f'{prefix}_{window_name}'] = vals

    return new_cols


In [8]:
def add_winsorized_variables(df, source_cols, quantiles, variable_notes=None):
    variable_notes = variable_notes if variable_notes is not None else []
    new_cols = []
    for col in source_cols:
        positive = df[col] > 0
        if positive.sum() == 0:
            continue
        for q in quantiles:
            q_label = int(q * 100)
            short_base = col.replace('ai_proximity_', '')
            new_col = safe_stata_varname(f'ai_w{q_label}_{short_base}')
            cap = df.loc[positive, col].quantile(q)
            df[new_col] = df[col]
            df.loc[positive & (df[new_col] > cap), new_col] = cap
            new_cols.append(new_col)
            variable_notes.append({
                'variable': new_col,
                'type': 'winsorized',
                'source_variable': col,
                'winsor_quantile_positive_tail': q,
                'cap_value': cap,
                'notes': 'positive DCE values above cap are set to the cap; zeros are unchanged'
            })
    return new_cols


def summarize_variables(df, cols):
    rows = []
    for col in cols:
        s = df[col]
        rows.append({
            'variable': col,
            'mean': s.mean(),
            'median': s.median(),
            'std': s.std(),
            'min': s.min(),
            'max': s.max(),
            'p90': s.quantile(0.90),
            'p95': s.quantile(0.95),
            'p99': s.quantile(0.99),
            'nonzero_n': int((s > 0).sum()),
            'nonzero_share': float((s > 0).mean()),
        })
    return pd.DataFrame(rows)


def build_temporal_windows(cutoff_year):
    prev_end = cutoff_year - 1
    prev_name = f"16_{str(prev_end)[-2:]}"
    post_name = f"{str(cutoff_year)[-2:]}_24"
    return {
        'before06': (None, 2005),
        '06_15': (2006, 2015),
        prev_name: (2016, prev_end),
        post_name: (cutoff_year, 2024),
    }


def check_larger_other_consistency(df, periods):
    checks = []
    for distance_km in BUFFER_DISTANCES:
        for period in periods:
            total_col = f'ai_proximity_{distance_km}km_{period}'
            larger_col = f'ai_larger_{distance_km}km_{period}'
            other_col = f'ai_otherdc_{distance_km}km_{period}'
            if all(c in df.columns for c in [total_col, larger_col, other_col]):
                diff = df[total_col] - df[larger_col] - df[other_col]
                checks.append({
                    'scope': f'{distance_km}km',
                    'period': period,
                    'total_variable': total_col,
                    'larger_variable': larger_col,
                    'other_variable': other_col,
                    'max_abs_difference': float(diff.abs().max()),
                })

    for gid in ['gid1', 'gid2', 'ezone']:
        for period in periods:
            total_col = f'ai_{gid}_{period}'
            larger_col = f'ai_larger_{gid}_{period}'
            other_col = f'ai_otherdc_{gid}_{period}'
            if all(c in df.columns for c in [total_col, larger_col, other_col]):
                diff = df[total_col] - df[larger_col] - df[other_col]
                checks.append({
                    'scope': 'Electricity zone' if gid == 'ezone' else gid.upper(),
                    'period': period,
                    'total_variable': total_col,
                    'larger_variable': larger_col,
                    'other_variable': other_col,
                    'max_abs_difference': float(diff.abs().max()),
                })

    checks_df = pd.DataFrame(checks)
    if len(checks_df) > 0:
        print("\nLarger + other consistency checks:")
        print(checks_df.to_string(index=False))
    return checks_df



def add_larger_other_classification_notes(variable_notes):
    prefix_notes = {}
    for spec in LARGER_OTHER_CLASSIFICATIONS:
        prefix_notes[spec['larger_prefix']] = spec['note']
        prefix_notes[spec['other_prefix']] = spec['note']
        prefix_notes[spec['cut_larger_prefix']] = spec['note']
        prefix_notes[spec['cut_other_prefix']] = spec['note']
        prefix_notes[spec['annual_larger_prefix']] = spec['note']
        prefix_notes[spec['annual_other_prefix']] = spec['note']

    for row in variable_notes:
        prefix = row.get('prefix', '')
        for pfx, note in prefix_notes.items():
            if prefix.startswith(pfx):
                row['larger_other_definition'] = note
                break
    return variable_notes


In [9]:
def process_dce_robust_capacity():
    print("Reading input data...")
    df_coal = pd.read_stata(COAL_INPUT)
    df_ai = pd.read_excel(AI_INPUT, sheet_name='Sheet1')
    df_ai = load_capacity_scenarios(df_ai)
    print(f"  coal records: {len(df_coal):,}")
    print(f"  raw AI/data center records: {len(df_ai):,}")

    df_ai['YR_BUILT'] = pd.to_numeric(df_ai['YR_BUILT'], errors='coerce')
    df_ai = df_ai.dropna(subset=['LATITUDE', 'LONGITUDE', 'YR_BUILT']).copy()
    df_ai = df_ai[df_ai['YR_BUILT'] != 2025].copy()
    df_ai['YR_BUILT'] = df_ai['YR_BUILT'].astype(int)
    df_coal = df_coal.dropna(subset=['Latitude', 'Longitude', 'year']).copy()

    print(f"  valid AI/data center records: {len(df_ai):,}")
    print(f"  valid coal records: {len(df_coal):,}")
    print(f"  AI year range: {df_ai['YR_BUILT'].min()}-{df_ai['YR_BUILT'].max()}")
    print(f"  Crypto mining records: {(df_ai['SECONDARY_PPTY_TYPE'] == CRYPTO_TYPE).sum():,}")
    print(f"  Wholesale records: {(df_ai['SECONDARY_PPTY_TYPE'] == WHOLESALE_TYPE).sum():,}")
    for spec in LARGER_OTHER_CLASSIFICATIONS:
        larger_n = df_ai['SECONDARY_PPTY_TYPE'].isin(spec['larger_types']).sum()
        print(f"  Larger records [{spec['label']}]: {larger_n:,}")
        print(f"  Other records  [{spec['label']}]: {len(df_ai) - larger_n:,}")

    df_coal, df_ai = ensure_gadm_attributes(df_coal, df_ai)
    df_coal, df_ai = ensure_electricity_zone_attributes(
        df_coal,
        df_ai,
    )
    df_coal, df_ai = add_region_keys(df_coal, df_ai)

    variable_notes = []
    all_new_cols = []

    all_new_cols += calculate_buffer_exposures(
        df_coal, df_ai, BUFFER_DISTANCES, BASE_WINDOWS,
        prefix='ai_proximity', min_distance_km=0.1,
        variable_notes=variable_notes
    )


    # ---------------------------------------------------------
    # Capacity-adjusted DCE sensitivity measures
    # ---------------------------------------------------------
    # Each scenario is calculated using:
    #   (1) capacity x inverse-distance weight; and
    #   (2) unweighted capacity summed within 25 km.
    # For each method, total, larger, and other exposure variables
    # are created for all four baseline periods and for all periods
    # combined.
    capacity_specs = []

    for scenario, capacity_col in CAPACITY_COLUMNS.items():
        method_specs = [
            {
                'method': 'distance_capacity',
                'total_prefix': f'ai_c{scenario}dw',
                'larger_prefix': f'ai_c{scenario}dwl',
                'other_prefix': f'ai_c{scenario}dwo',
            },
            {
                'method': 'capacity_sum',
                'total_prefix': f'ai_c{scenario}cs',
                'larger_prefix': f'ai_c{scenario}csl',
                'other_prefix': f'ai_c{scenario}cso',
            },
        ]

        for method_spec in method_specs:
            method_spec['capacity_col'] = capacity_col
            capacity_specs.append(method_spec.copy())

            all_new_cols += calculate_capacity_buffer_exposures(
                df_coal=df_coal,
                df_ai=df_ai,
                capacity_col=capacity_col,
                time_windows=CAPACITY_WINDOWS,
                prefix=method_spec['total_prefix'],
                method=method_spec['method'],
                buffer_km=CAPACITY_BUFFER_KM,
                min_distance_km=CAPACITY_MIN_DISTANCE_KM,
                variable_notes=variable_notes,
            )

            all_new_cols += calculate_capacity_buffer_exposures(
                df_coal=df_coal,
                df_ai=df_ai,
                capacity_col=capacity_col,
                time_windows=CAPACITY_WINDOWS,
                prefix=method_spec['larger_prefix'],
                method=method_spec['method'],
                buffer_km=CAPACITY_BUFFER_KM,
                min_distance_km=CAPACITY_MIN_DISTANCE_KM,
                ai_filter=lambda x: x['SECONDARY_PPTY_TYPE'].isin(
                    LARGER_DC_TYPES
                ),
                variable_notes=variable_notes,
            )

            all_new_cols += calculate_capacity_buffer_exposures(
                df_coal=df_coal,
                df_ai=df_ai,
                capacity_col=capacity_col,
                time_windows=CAPACITY_WINDOWS,
                prefix=method_spec['other_prefix'],
                method=method_spec['method'],
                buffer_km=CAPACITY_BUFFER_KM,
                min_distance_km=CAPACITY_MIN_DISTANCE_KM,
                ai_filter=lambda x: ~x['SECONDARY_PPTY_TYPE'].isin(
                    LARGER_DC_TYPES
                ),
                variable_notes=variable_notes,
            )

    capacity_split_checks = check_capacity_larger_other_consistency(
        df_coal,
        capacity_specs,
        CAPACITY_WINDOWS.keys(),
    )
    if len(capacity_split_checks) > 0:
        capacity_check_output = os.path.join(
            TEMP,
            'dce_robust3_capacity_larger_other_consistency.csv'
        )
        capacity_split_checks.to_csv(
            capacity_check_output,
            index=False
        )
        print(
            "Saved capacity larger/other consistency checks: "
            f"{capacity_check_output}"
        )

    # Larger vs. other data-center decomposition for all four baseline periods.
    # Includes the original split and two alternative classification rules.
    for spec in LARGER_OTHER_CLASSIFICATIONS:
        larger_types = spec['larger_types']
        all_new_cols += calculate_buffer_exposures(
            df_coal, df_ai, BUFFER_DISTANCES, BASE_WINDOWS,
            prefix=spec['larger_prefix'], min_distance_km=0.1,
            ai_filter=lambda x, lt=larger_types: x['SECONDARY_PPTY_TYPE'].isin(lt),
            variable_notes=variable_notes
        )

        all_new_cols += calculate_buffer_exposures(
            df_coal, df_ai, BUFFER_DISTANCES, BASE_WINDOWS,
            prefix=spec['other_prefix'], min_distance_km=0.1,
            ai_filter=lambda x, lt=larger_types: ~x['SECONDARY_PPTY_TYPE'].isin(lt),
            variable_notes=variable_notes
        )

    all_new_cols += calculate_buffer_exposures(
        df_coal, df_ai, BUFFER_DISTANCES, BASE_WINDOWS,
        prefix='ai_nocrypto', min_distance_km=0.1,
        ai_filter=lambda x: x['SECONDARY_PPTY_TYPE'] != CRYPTO_TYPE,
        variable_notes=variable_notes
    )

    all_new_cols += calculate_buffer_exposures(
        df_coal, df_ai, BUFFER_DISTANCES, BASE_WINDOWS,
        prefix='ai_nopre00', min_distance_km=0.1,
        ai_filter=lambda x: x['YR_BUILT'] >= 2000,
        variable_notes=variable_notes
    )

    # Temporal cutoff robustness: total DCE plus larger/other decompositions.
    # Cutoff variables are calculated only for TEMPORAL_CUTOFF_BUFFERS, currently 25 km.
    for cutoff in TEMPORAL_CUTOFFS:
        cutoff_suffix = str(cutoff)[-2:]
        windows = build_temporal_windows(cutoff)

        all_new_cols += calculate_buffer_exposures(
            df_coal, df_ai, TEMPORAL_CUTOFF_BUFFERS, windows,
            prefix=f'ai_cut{cutoff_suffix}', min_distance_km=0.1,
            variable_notes=variable_notes
        )

        for spec in LARGER_OTHER_CLASSIFICATIONS:
            larger_types = spec['larger_types']
            all_new_cols += calculate_buffer_exposures(
                df_coal, df_ai, TEMPORAL_CUTOFF_BUFFERS, windows,
                prefix=f"{spec['cut_larger_prefix']}{cutoff_suffix}", min_distance_km=0.1,
                ai_filter=lambda x, lt=larger_types: x['SECONDARY_PPTY_TYPE'].isin(lt),
                variable_notes=variable_notes
            )

            all_new_cols += calculate_buffer_exposures(
                df_coal, df_ai, TEMPORAL_CUTOFF_BUFFERS, windows,
                prefix=f"{spec['cut_other_prefix']}{cutoff_suffix}", min_distance_km=0.1,
                ai_filter=lambda x, lt=larger_types: ~x['SECONDARY_PPTY_TYPE'].isin(lt),
                variable_notes=variable_notes
            )

    # Annual timing robustness: one mutually exclusive multi-period
    # specification isolates facilities commissioned in each year from
    # 2018 through 2022. It is calculated for all, larger, and other data
    # centers using the same 25 km inverse-distance-weighted DCE definition.
    all_new_cols += calculate_buffer_exposures(
        df_coal, df_ai, ANNUAL_WINDOW_BUFFERS, ANNUAL_WINDOWS,
        prefix='ai_annual', min_distance_km=0.1,
        variable_notes=variable_notes
    )

    for spec in LARGER_OTHER_CLASSIFICATIONS:
        larger_types = spec['larger_types']
        all_new_cols += calculate_buffer_exposures(
            df_coal, df_ai, ANNUAL_WINDOW_BUFFERS, ANNUAL_WINDOWS,
            prefix=spec['annual_larger_prefix'],
            min_distance_km=0.1,
            ai_filter=lambda x, lt=larger_types: x[
                'SECONDARY_PPTY_TYPE'
            ].isin(lt),
            variable_notes=variable_notes
        )

        all_new_cols += calculate_buffer_exposures(
            df_coal, df_ai, ANNUAL_WINDOW_BUFFERS, ANNUAL_WINDOWS,
            prefix=spec['annual_other_prefix'],
            min_distance_km=0.1,
            ai_filter=lambda x, lt=larger_types: ~x[
                'SECONDARY_PPTY_TYPE'
            ].isin(lt),
            variable_notes=variable_notes
        )

    for cap in MIN_DISTANCE_CAPS_KM:
        all_new_cols += calculate_buffer_exposures(
            df_coal, df_ai, MIN_CAP_BUFFERS, BASE_WINDOWS,
            prefix=f'ai_cap{cap}km', min_distance_km=cap,
            variable_notes=variable_notes
        )

    region_specs = [
        ('ai_gid1', 'key_gid1', 'key_gid1'),
        ('ai_gid2', 'key_gid2', 'key_gid2'),
        ('ai_ezone', 'key_ezone', 'key_ezone'),
    ]
    all_new_cols += calculate_region_exposures(
        df_coal, df_ai, region_specs, BASE_WINDOWS,
        min_distance_km=0.1,
        variable_notes=variable_notes
    )

    # Same-region larger/other decomposition under each classification rule.
    for spec in LARGER_OTHER_CLASSIFICATIONS:
        larger_types = spec['larger_types']
        larger_region_specs = [
            (f"{spec['larger_prefix']}_gid1", 'key_gid1', 'key_gid1'),
            (f"{spec['larger_prefix']}_gid2", 'key_gid2', 'key_gid2'),
        ]
        if spec['label'] == 'baseline':
            larger_region_specs.append(
                ('ai_larger_ezone', 'key_ezone', 'key_ezone')
            )
        all_new_cols += calculate_region_exposures(
            df_coal, df_ai, larger_region_specs, BASE_WINDOWS,
            min_distance_km=0.1,
            ai_filter=lambda x, lt=larger_types: x['SECONDARY_PPTY_TYPE'].isin(lt),
            variable_notes=variable_notes
        )

        other_region_specs = [
            (f"{spec['other_prefix']}_gid1", 'key_gid1', 'key_gid1'),
            (f"{spec['other_prefix']}_gid2", 'key_gid2', 'key_gid2'),
        ]
        if spec['label'] == 'baseline':
            other_region_specs.append(
                ('ai_otherdc_ezone', 'key_ezone', 'key_ezone')
            )
        all_new_cols += calculate_region_exposures(
            df_coal, df_ai, other_region_specs, BASE_WINDOWS,
            min_distance_km=0.1,
            ai_filter=lambda x, lt=larger_types: ~x['SECONDARY_PPTY_TYPE'].isin(lt),
            variable_notes=variable_notes
        )

    split_checks = check_larger_other_consistency(df_coal, BASE_WINDOWS.keys())
    if len(split_checks) > 0:
        check_output = os.path.join(TEMP, 'dce_robust3_larger_other_consistency.csv')
        split_checks.to_csv(check_output, index=False)
        print(f"Saved larger/other consistency checks: {check_output}")

    baseline_cols = [c for c in all_new_cols if c.startswith('ai_proximity_')]
    all_new_cols += add_winsorized_variables(
        df_coal, baseline_cols, WINSOR_QUANTILES,
        variable_notes=variable_notes
    )

    variable_notes = add_larger_other_classification_notes(variable_notes)

    var_dict = pd.DataFrame(variable_notes).drop_duplicates(subset=['variable'])
    var_dict.to_csv(VAR_DICT_OUTPUT, index=False)
    print(f"\nSaved variable dictionary: {VAR_DICT_OUTPUT}")

    summary = summarize_variables(df_coal, all_new_cols)
    summary.to_csv(SUMMARY_OUTPUT, index=False)
    print(f"Saved summary statistics: {SUMMARY_OUTPUT}")

    print(f"\nSaving robust3 DCE dataset including capacity-adjusted measures with {len(all_new_cols)} new variables...")
    df_coal.to_stata(OUTPUT_DTA, write_index=False, version=118)
    print(f"Saved robust3 DCE dataset: {OUTPUT_DTA}")

    return df_coal, all_new_cols, var_dict, summary


## Capacity-adjusted variable naming

For each capacity scenario `c1`-`c6`:

| Prefix | Definition |
|---|---|
| `ai_c#dw` | all data centers: capacity multiplied by inverse-distance weight |
| `ai_c#dwl` | larger data centers: capacity multiplied by inverse-distance weight |
| `ai_c#dwo` | other data centers: capacity multiplied by inverse-distance weight |
| `ai_c#cs` | all data centers: capacity summed within 25 km |
| `ai_c#csl` | larger data centers: capacity summed within 25 km |
| `ai_c#cso` | other data centers: capacity summed within 25 km |

Every prefix is followed by `25km` and one of `before06`, `06_15`,
`16_19`, `20_24`, or `all`. For example,
`ai_c2dwl_25km_20_24` is larger-data-center exposure during 2020-2024
under capacity scenario 2, weighted jointly by capacity and inverse distance;
`ai_c2csl_25km_all` is the corresponding capacity sum across all
commissioning periods.


## Annual timing variables

The moving-cutoff variables are retained unchanged. One additional
multi-period specification divides commissioning dates into `before06`,
`06_15`, `16_17`, the individual years `2018`, `2019`, `2020`, `2021`,
and `2022`, and the final period `23_24`.

Variable prefixes are:

| Prefix | Definition |
|---|---|
| `ai_annual` | all data centers |
| `ai_lgann` / `ai_otann` | baseline larger/other classification |
| `ai_lnann` / `ai_onann` | crypto mining moved to other |
| `ai_lsann` / `ai_osann` | only hyperscale/cloud classified as larger |

All single-year timing variables use the 25 km inverse-distance-weighted DCE
definition and the baseline 100 m minimum-distance cap.


## Electricity-zone exposure

Electricity-zone exposure is calculated for all data centers
(`ai_ezone_*`) and, under the baseline classification, separately for larger
(`ai_larger_ezone_*`) and other (`ai_otherdc_ezone_*`) data centers. Each set
contains the four baseline commissioning periods: `before06`, `06_15`,
`16_19`, and `20_24`. The measures use Electricity Maps zone boundaries from
`Data/raw/world.geojson`. For each coal unit, exposure is the
inverse-distance-weighted sum of operational data centers located in the same
electricity zone, without a circular buffer.
If zone polygons overlap, the smallest-area matching polygon is retained.
Countries represented by a single national electricity zone are used as
provided by the source data.

The standard spatial sensitivity set uses 15, 25, 50, 100, and 200 km
buffers.


In [ ]:
# Run robust3 DCE calculation with capacity-adjusted measures.
# This can take a while because it computes distances from each coal unit to all valid data centers.

if __name__ == "__main__":
    try:
        robust_df, robust_cols, var_dict, summary = process_dce_robust_capacity()
        print("\nDone.")
        print(f"Total robust3 and capacity-adjusted DCE variables: {len(robust_cols)}")
        print(summary.head(20).to_string(index=False))
    except KeyboardInterrupt:
        print("\nExecution interrupted by user")
    except Exception:
        import traceback
        traceback.print_exc()


## Notes on robustness design

This notebook intentionally uses a one-factor-at-a-time robustness design. For example, `ai_nocrypto_*` changes only the crypto-mining inclusion rule while keeping the original distance cap and time windows; `ai_cap1km_*` changes only the minimum-distance cap; `ai_cut21_*` changes only the temporal cutoff.

A full factorial design, such as `no crypto x 200 km x 1 km cap x cutoff 2021 x no pre-2000`, is usually not needed for reviewer response and would make the regression tables difficult to interpret. If a key result is fragile, a small number of targeted joint checks can be added later.


In [ ]:
# Diagnostics: how many raw coal-unit/data-center distances fall below the minimum-distance caps?
# This code does not apply any distance cap. It uses raw haversine distances.

import os
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm.auto import tqdm

# ---------------------------------------------------------------------
# 1. Load and clean data exactly as in dce_calculate_robust3
# ---------------------------------------------------------------------

from pathlib import Path

try:
    COAL_INPUT
    AI_INPUT
    TEMP
    TABLES
except NameError:
    BASE_PATH = Path.cwd().resolve()

    while (
        BASE_PATH.name != "Data_center_and_fossil_energy_Replication"
        and BASE_PATH.parent != BASE_PATH
    ):
        BASE_PATH = BASE_PATH.parent

    if BASE_PATH.name != "Data_center_and_fossil_energy_Replication":
        raise RuntimeError(
            "Project root 'Data_center_and_fossil_energy_Replication' "
            "could not be located."
        )

    RAW = BASE_PATH / "Data" / "raw"
    TEMP = BASE_PATH / "Data" / "temp"
    TABLES = BASE_PATH / "Results" / "Tables"

    COAL_INPUT = (
        TEMP / "gem_coal_plants_multi_record_sa_sinceoperating.dta"
    )
    AI_INPUT = RAW / "SPGlobal_Export.xlsx"

os.makedirs(TEMP, exist_ok=True)
os.makedirs(TABLES, exist_ok=True)

BASE_WINDOWS = {
    'before06': (None, 2005),
    '06_15': (2006, 2015),
    '16_19': (2016, 2019),
    '20_24': (2020, 2024),
}

thresholds_km = [0.1, 1, 2, 3]
threshold_labels = {
    0.1: "100 m",
    1: "1 km",
    2: "2 km",
    3: "3 km",
}

def calculate_distance_vectorized(lat1, lon1, lat2_array, lon2_array):
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2_array)
    lon2_rad = np.radians(lon2_array)
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c

def time_window_mask(ai_year, start_year, end_year):
    if start_year is None:
        return ai_year <= end_year
    return (ai_year >= start_year) & (ai_year <= end_year)

df_coal = pd.read_stata(COAL_INPUT)
df_ai = pd.read_excel(AI_INPUT, sheet_name="Sheet1")

df_ai["YR_BUILT"] = pd.to_numeric(df_ai["YR_BUILT"], errors="coerce")
df_ai = df_ai.dropna(subset=["LATITUDE", "LONGITUDE", "YR_BUILT"]).copy()
df_ai = df_ai[df_ai["YR_BUILT"] != 2025].copy()
df_ai["YR_BUILT"] = df_ai["YR_BUILT"].astype(int)

df_coal = df_coal.dropna(subset=["Latitude", "Longitude", "year"]).copy()
df_coal["year"] = df_coal["year"].astype(int)

unique_units = (
    df_coal[["GEM_unit_phase_ID", "Latitude", "Longitude"]]
    .drop_duplicates("GEM_unit_phase_ID")
    .copy()
)

unit_years = (
    df_coal
    .groupby("GEM_unit_phase_ID", sort=False)["year"]
    .apply(lambda s: s.astype(int).to_numpy())
    .to_dict()
)

ai_lat = df_ai["LATITUDE"].to_numpy(float)
ai_lon = df_ai["LONGITUDE"].to_numpy(float)
ai_year = df_ai["YR_BUILT"].to_numpy(int)

ai_id_col = "PPTY_KEY" if "PPTY_KEY" in df_ai.columns else None
ai_type_col = "SECONDARY_PPTY_TYPE" if "SECONDARY_PPTY_TYPE" in df_ai.columns else None
ai_name_col = "PPTY_NAME" if "PPTY_NAME" in df_ai.columns else None
ai_country_col = "COUNTRY" if "COUNTRY" in df_ai.columns else None

n_units = len(unique_units)
n_ai = len(df_ai)
n_panel = len(df_coal)
total_possible_pairs = n_units * n_ai

print(f"Unique coal units: {n_units:,}")
print(f"Valid data centers: {n_ai:,}")
print(f"Coal-unit-year panel records: {n_panel:,}")
print(f"Total possible unique coal-unit/data-center pairs: {total_possible_pairs:,}")

# ---------------------------------------------------------------------
# 2. Count raw distances below thresholds
# ---------------------------------------------------------------------

pair_counts = defaultdict(int)
unit_sets = defaultdict(set)
dc_sets = defaultdict(set)

period_pair_counts = defaultdict(int)

panel_record_counts = defaultdict(int)
panel_unit_sets = defaultdict(set)

nearest_rows = []
close_pair_rows = []

max_threshold = max(thresholds_km)

for _, unit in tqdm(unique_units.iterrows(), total=n_units, desc="Raw distance diagnostics", ncols=90):
    gem_id = unit["GEM_unit_phase_ID"]
    coal_lat = float(unit["Latitude"])
    coal_lon = float(unit["Longitude"])

    target_years = unit_years[gem_id]

    distances_km = calculate_distance_vectorized(coal_lat, coal_lon, ai_lat, ai_lon)

    nearest_idx = int(np.argmin(distances_km))
    nearest_rows.append({
        "GEM_unit_phase_ID": gem_id,
        "nearest_dc_distance_km": distances_km[nearest_idx],
        "nearest_dc_year_built": ai_year[nearest_idx],
        "nearest_dc_key": df_ai.iloc[nearest_idx][ai_id_col] if ai_id_col else nearest_idx,
        "nearest_dc_type": df_ai.iloc[nearest_idx][ai_type_col] if ai_type_col else "",
        "nearest_dc_name": df_ai.iloc[nearest_idx][ai_name_col] if ai_name_col else "",
        "nearest_dc_country": df_ai.iloc[nearest_idx][ai_country_col] if ai_country_col else "",
    })

    close_to_max = distances_km <= max_threshold

    if close_to_max.any():
        close_idx = np.where(close_to_max)[0]
        for j in close_idx:
            close_pair_rows.append({
                "GEM_unit_phase_ID": gem_id,
                "coal_latitude": coal_lat,
                "coal_longitude": coal_lon,
                "dc_index": int(j),
                "dc_key": df_ai.iloc[j][ai_id_col] if ai_id_col else int(j),
                "dc_name": df_ai.iloc[j][ai_name_col] if ai_name_col else "",
                "dc_country": df_ai.iloc[j][ai_country_col] if ai_country_col else "",
                "dc_type": df_ai.iloc[j][ai_type_col] if ai_type_col else "",
                "dc_year_built": int(ai_year[j]),
                "raw_distance_km": float(distances_km[j]),
                "within_100m": bool(distances_km[j] <= 0.1),
                "within_1km": bool(distances_km[j] <= 1),
                "within_2km": bool(distances_km[j] <= 2),
                "within_3km": bool(distances_km[j] <= 3),
            })

    for th in thresholds_km:
        within_th = distances_km <= th
        close_n = int(within_th.sum())

        pair_counts[th] += close_n

        if close_n > 0:
            unit_sets[th].add(gem_id)
            dc_sets[th].update(np.where(within_th)[0].tolist())

        # Pair counts by commissioning period
        for period, (start_year, end_year) in BASE_WINDOWS.items():
            period_mask = time_window_mask(ai_year, start_year, end_year)
            period_pair_counts[(th, period)] += int((within_th & period_mask).sum())

        # Panel-record counts:
        # A coal-unit-year record is counted if at least one data center within threshold
        # and within the commissioning period has already been built by that panel year.
        for period, (start_year, end_year) in BASE_WINDOWS.items():
            valid = within_th & time_window_mask(ai_year, start_year, end_year)

            if not valid.any():
                continue

            built_years_sorted = np.sort(ai_year[valid])
            active_record = np.searchsorted(
                built_years_sorted,
                target_years,
                side="right"
            ) > 0

            active_n = int(active_record.sum())
            panel_record_counts[(th, period)] += active_n

            if active_n > 0:
                panel_unit_sets[(th, period)].add(gem_id)

# ---------------------------------------------------------------------
# 3. Build summary tables
# ---------------------------------------------------------------------

pair_summary = []
for th in thresholds_km:
    pair_summary.append({
        "threshold_km": th,
        "threshold_label": threshold_labels[th],
        "raw_pair_count": pair_counts[th],
        "raw_pair_share_of_all_possible_pairs": pair_counts[th] / total_possible_pairs,
        "unique_coal_units_with_at_least_one_dc": len(unit_sets[th]),
        "unique_coal_unit_share": len(unit_sets[th]) / n_units,
        "unique_data_centers_involved": len(dc_sets[th]),
        "unique_data_center_share": len(dc_sets[th]) / n_ai,
    })

pair_summary = pd.DataFrame(pair_summary)

period_pair_summary = []
for th in thresholds_km:
    for period in BASE_WINDOWS:
        period_pair_summary.append({
            "threshold_km": th,
            "threshold_label": threshold_labels[th],
            "period": period,
            "raw_pair_count": period_pair_counts[(th, period)],
            "raw_pair_share_of_all_possible_pairs": period_pair_counts[(th, period)] / total_possible_pairs,
        })

period_pair_summary = pd.DataFrame(period_pair_summary)

panel_record_summary = []
for th in thresholds_km:
    for period in BASE_WINDOWS:
        panel_record_summary.append({
            "threshold_km": th,
            "threshold_label": threshold_labels[th],
            "period": period,
            "panel_records_with_active_close_dc": panel_record_counts[(th, period)],
            "panel_record_share": panel_record_counts[(th, period)] / n_panel,
            "unique_coal_units_with_active_close_dc": len(panel_unit_sets[(th, period)]),
            "unique_coal_unit_share": len(panel_unit_sets[(th, period)]) / n_units,
        })

panel_record_summary = pd.DataFrame(panel_record_summary)

nearest_distance = pd.DataFrame(nearest_rows)
close_pair_details = pd.DataFrame(close_pair_rows)

nearest_summary = pd.DataFrame([
    {
        "threshold_km": th,
        "threshold_label": threshold_labels[th],
        "unique_coal_units_with_nearest_dc_within_threshold": int((nearest_distance["nearest_dc_distance_km"] <= th).sum()),
        "share_of_unique_coal_units": float((nearest_distance["nearest_dc_distance_km"] <= th).mean()),
    }
    for th in thresholds_km
])

# ---------------------------------------------------------------------
# 4. Save outputs
# ---------------------------------------------------------------------

pair_summary_path = os.path.join(TABLES, "dce_raw_distance_cap_pair_summary.csv")
period_pair_summary_path = os.path.join(TABLES, "dce_raw_distance_cap_pair_summary_by_period.csv")
panel_record_summary_path = os.path.join(TABLES, "dce_raw_distance_cap_panel_record_summary_by_period.csv")
nearest_summary_path = os.path.join(TABLES, "dce_raw_distance_cap_nearest_unit_summary.csv")
nearest_distance_path = os.path.join(TEMP, "dce_raw_distance_nearest_dc_by_coal_unit.csv")
close_pair_details_path = os.path.join(TEMP, "dce_raw_distance_close_pairs_within_3km.csv")

pair_summary.to_csv(pair_summary_path, index=False)
period_pair_summary.to_csv(period_pair_summary_path, index=False)
panel_record_summary.to_csv(panel_record_summary_path, index=False)
nearest_summary.to_csv(nearest_summary_path, index=False)
nearest_distance.to_csv(nearest_distance_path, index=False)
close_pair_details.to_csv(close_pair_details_path, index=False)

# ---------------------------------------------------------------------
# 5. Print results
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("Raw distance cap diagnostics: unique coal-unit/data-center pairs")
print("=" * 80)
display(pair_summary)

print("\n" + "=" * 80)
print("Raw close-pair counts by data-center commissioning period")
print("=" * 80)
display(period_pair_summary.pivot(index="period", columns="threshold_label", values="raw_pair_count"))

print("\n" + "=" * 80)
print("Coal-unit-year panel records with at least one active close data center")
print("=" * 80)
display(panel_record_summary.pivot(index="period", columns="threshold_label", values="panel_records_with_active_close_dc"))

print("\n" + "=" * 80)
print("Unique coal units whose nearest data center is within each threshold")
print("=" * 80)
display(nearest_summary)

print("\nSaved outputs:")
print(pair_summary_path)
print(period_pair_summary_path)
print(panel_record_summary_path)
print(nearest_summary_path)
print(nearest_distance_path)
print(close_pair_details_path)
print("=" * 80)